In [1]:
import torch
import torch.nn.functional as F

In [46]:
def model_params(dm, n_laers, dff=4, vs=50000):
    embedding = vs*dm
    att = (dm**2)*4
    ff = (dm*dm*dff)*2
    block = att+ff
    ap = n_laers*block+2*embedding
    return ap, 2*embedding

def pruned_model_params(dm, doner_dm, n_laers, dff=4, vs=50000):
    embedding = vs*dm
    att = (dm*doner_dm)*4
    ff = (dm*dm*dff)*2
    block = att+ff
    ap = n_laers*block+2*embedding
    return ap, 2*embedding

def compressor_active_parameters(dm, doner_dm, n_laers, dff=4, vs=50000):
    residual_model, res_emb = pruned_model_params(dm, doner_dm, n_laers, dff, vs)
    # frozen_model = model_params(doner_dm, n_laers, dff, vs) # dev frozen parameteres
    att_p = (dm*doner_dm)*4
    ff_p = (dm*doner_dm)*2 + (dm*dff*doner_dm*dff)*2
    projections_param = (att_p + ff_p) * n_laers + 2*(dm*doner_dm)
    # total_active = residual_model + projections_param
    total_active =  projections_param
    return total_active, (2*(dm*doner_dm))

def calculate_total_steps(parameters, tokens_step=512*512, tpp=20):
    return  (tpp*parameters)/(tokens_step)

def create_description(doner_dm, pruned_dm, n_layers, dff=4):
    assert doner_dm%n_layers==0
    assert pruned_dm%n_layers==0
    doner_ap, doner_emb = model_params(doner_dm, n_layers, dff)
    doner_s = calculate_total_steps(doner_ap)
    pruned_ap, pruned_emb = pruned_model_params(pruned_dm, doner_dm, n_layers, dff)
    pruned_s = calculate_total_steps(pruned_ap)
    compressor_ap, compressor_emb = compressor_active_parameters(pruned_dm, doner_dm, n_layers, dff)
    compressor_s = calculate_total_steps(compressor_ap)
    compresor_total_s = calculate_total_steps(compressor_ap+doner_ap)

    print(f"Doner dmodel: {doner_dm}, prunned dm: {pruned_dm}, n_layters: {n_layers}")
    print(f"Doner model [{round(doner_ap/10**6, 2)}, {round(doner_emb/10**6, 2)}], opt. steps: {doner_s}")
    print(f"Pruned model [{round(pruned_ap/10**6, 2)}, {round(pruned_emb/10**6, 2)}], opt. steps: {pruned_s}")
    print(f"Compression ratio: {round(pruned_ap/doner_ap, 2)}")
    print(f"Compressor model [{round(compressor_ap/10**6)}, {round(compressor_emb/10**6, 2)}], opt. steps: {compressor_s}")
    print(f"Compressor model all params [{round((compressor_ap+doner_ap)/10**6, 2)}, {round((compressor_emb+doner_emb)/10**6, 2)}], opt. steps: {compresor_total_s}")
    print(f"Compression RATIO: {round(doner_ap / pruned_ap, 2)}, Sparsity: {round((doner_ap-pruned_ap)/doner_ap, 2)}")
    print(f"Compression TOWER RATIO: {round((doner_ap-doner_emb) / (pruned_ap-pruned_emb), 2)}, Sparsity: {round(((doner_ap-doner_emb)-(pruned_ap-pruned_emb))/(doner_ap-doner_emb), 2)}")

DONER_DM = 1024
DONER_DM_L = 1024*1.5

N_LAYERS = 16
N_LAYERS_L = 16*1.5

DFF = 4

In [9]:
create_description(DONER_DM, DONER_DM*3/4, N_LAYERS, DFF)


Doner dmodel: 1024, prunned dm: 768.0, n_layters: 16
Doner model [303.73, 102.4], opt. steps: 23172.5
Pruned model [202.63, 76.8], opt. steps: 15459.375
Compression ratio: 0.67
Compressor model [480, 1.57], opt. steps: 36600.0
Compressor model all params [783.45, 103.97], opt. steps: 59772.5
Compression RATIO: 1.5, Sparsity: 0.33
Compression TOWER RATIO: 1.6


In [48]:
create_description(DONER_DM_L, DONER_DM_L*24/32, N_LAYERS_L, DFF)


Doner dmodel: 1536.0, prunned dm: 1152.0, n_layters: 24.0
Doner model [833.08, 153.6], opt. steps: 63558.75
Pruned model [539.87, 115.2], opt. steps: 41189.0625
Compression ratio: 0.65
Compressor model [1617, 3.54], opt. steps: 123390.0
Compressor model all params [2450.37, 157.14], opt. steps: 186948.75
Compression RATIO: 1.54, Sparsity: 0.35
Compression TOWER RATIO: 1.6, Sparsity: 0.38


In [57]:
create_description(DONER_DM, 832, N_LAYERS, 1)


Doner dmodel: 1024, prunned dm: 832, n_layters: 16
Doner model [203.06, 102.4], opt. steps: 15492.5
Pruned model [159.88, 83.2], opt. steps: 12197.65625
Compression ratio: 0.79
Compressor model [111, 1.7], opt. steps: 8450.0
Compressor model all params [313.82, 104.1], opt. steps: 23942.5
Compression RATIO: 1.27, Sparsity: 0.21
Compression TOWER RATIO: 1.31, Sparsity: 0.24


In [49]:

create_description(1536, 768, 24, 4)
print("")
create_description(1536, 960, 24, 4)
print("\nERRONEOUS 32 layers on nemo side")
create_description(1536, 768, 32, 4)


print("\nERRONEOUS 16/32 layers comparison")
doner_ap_n, doner_emb_n = model_params(1536, 32)
doner_ap_llmr, doner_emb_llmr = model_params(1536, 24)

pruned_ap_n, pruned_emb_n = pruned_model_params(768, 1536, 32)
pruned_ap_llmr, pruned_emb_llmr = pruned_model_params(768, 1536, 24)
print(f"Doner comparison: {round(doner_ap_n/10**6,2)} ; {round(doner_ap_llmr/10**6,2)}, ratio: {round(doner_ap_n/doner_ap_llmr,2)}")
print(f"Prunned comparison: {round(pruned_ap_n/10**6,2)} ; {round(pruned_ap_llmr/10**6,2)}, ratio: {round(pruned_ap_n/pruned_ap_llmr,2)}")


Doner dmodel: 1536, prunned dm: 768, n_layters: 24
Doner model [833.08, 153.6], opt. steps: 63558.75
Pruned model [303.29, 76.8], opt. steps: 23139.375
Compression ratio: 0.36
Compressor model [1078, 2.36], opt. steps: 82260.0
Compressor model all params [1911.28, 155.96], opt. steps: 145818.75
Compression RATIO: 2.75, Sparsity: 0.64
Compression TOWER RATIO: 3.0, Sparsity: 0.67

Doner dmodel: 1536, prunned dm: 960, n_layters: 24
Doner model [833.08, 153.6], opt. steps: 63558.75
Pruned model [414.5, 96.0], opt. steps: 31624.21875
Compression ratio: 0.5
Compressor model [1348, 2.95], opt. steps: 102825.0
Compressor model all params [2180.83, 156.55], opt. steps: 166383.75
Compression RATIO: 2.01, Sparsity: 0.5
Compression TOWER RATIO: 2.13, Sparsity: 0.53

ERRONEOUS 32 layers on nemo side
Doner dmodel: 1536, prunned dm: 768, n_layters: 32
Doner model [1059.57, 153.6], opt. steps: 80838.75
Pruned model [378.79, 76.8], opt. steps: 28899.375
Compression ratio: 0.36
Compressor model [1437, 2

In [ ]:
pruned_ap_llmr, pruned_emb_llmr = pruned_model_params(768, 1536, 24)
(pruned_ap_llmr - pruned_emb_llmr)/10**6

226.492416

In [ ]:
embedding = 50432*768
att = (768*1536)*4
ff = (768*3072)*2
block = att+ff
ap = 24*block+2*embedding
ap, 2*embedding, ap-2*embedding

(303955968, 77463552, 226492416)

In [ ]:
US = 20000
CR = 0.5
def calculate_compresor_flops_ratio(dm, doner_dm, n_layers, dff=4, vs=50000):
    compressor_ap, compressor_emb = compressor_active_parameters(dm, doner_dm, n_layers, dff, vs)
    doner_dm_fp, doner_dm_fp_emb = model_params(doner_dm, n_layers, dff, vs)
    pruned_ap, pruned_emb = pruned_model_params(dm, doner_dm, n_layers, dff, vs)
    
    compressor_ap = compressor_ap-compressor_emb
    doner_dm_fp = doner_dm_fp-doner_dm_fp_emb
    pruned_ap = pruned_ap-pruned_emb

    pruned_cost = pruned_ap*3
    # compressor_cost = pruned_ap + (pruned_ap+compressor_ap+doner_dm_fp)*2
    # compressor_cost = pruned_ap + (pruned_ap+compressor_ap)*2
    # compressor_cost = pruned_ap + (pruned_ap+compressor_ap)*2 + (dm*doner_dm*5 + (doner_dm*dm*dff*dff))*n_layers #  + (doner_dm*doner_dm)*4*n_layers
    # compressor_cost = (pruned_ap+compressor_ap+doner_dm_fp) + (pruned_ap+compressor_ap)*2

    compressor_cost = pruned_ap + (compressor_ap)*2 + (dm*doner_dm*5 + (doner_dm*dm*dff*dff))*n_layers
    
    return compressor_cost/pruned_cost, compressor_cost, pruned_cost

calculate_compresor_flops_ratio(DM, DONER_DM, N_LAYERS, DFF)# , calculate_compresor_flops_ratio(DONER_DM*CR, DONER_DM, N_LAYERS, DFF)#, calculate_compresor_flops_ratio(DM*US, DONER_DM*US, N_LAYERS*US, DFF)


(3.566666666666667, 1346371584, 377487360)

In [ ]:
for i in range(5):
    print(i*31624)

0
31624
63248
94872
126496


In [ ]:
N_LAYERS = 24
DM = N_LAYERS*64

def model_params_1ff(dm, n_laers, vs=50000):
    embedding = vs*dm
    block = (dm**2)*6
    ap = n_laers*block+2*embedding
    return ap, 2*embedding

def model_params_1ff_compressor(dm, dm_projected, n_laers=8, vs=50000):
    projection = dm*dm_projected
    embedding = projection
    block = projection*12
    ap = n_laers*block+3*embedding
    return ap, 3*embedding

def minitron_params_1ff(dm, n_laers=8, att_ratio=1.0, doner_dm=1024, vs=50000):
    embedding = vs*dm
    block_1 = (dm*dm)*2
    block_2 = (dm*doner_dm)*4
    block = block_1+block_2*att_ratio
    # block = (dm**2)*6
    ap = n_laers*block+2*embedding
    return ap, 2*embedding

# def_t, def_e = model_params_1ff(768, 16)
# min_t, min_e = minitron_params_1ff(768, 16)

# def_t-def_e, min_t-min_e, 0.75*16
# min_t, min_e, def_t, def_e

def compress_info(doner_dm, compressed_dm, n_layers):
    compressor_ap, compressor_ap_e = model_params_1ff_compressor(doner_dm, compressed_dm, n_layers)
    default_ap, default_ap_e = model_params_1ff(compressed_dm, n_layers)
    doner_ap, doner_ap_e = model_params_1ff(doner_dm, n_layers)

    print(f"N layers: {n_layers}, doner dmodel: {doner_dm}, compressed dmodel: {compressed_dm}, head/layers: {compressed_dm/n_layers}")
    print("[Model]: [Act. params] - [Embedding act. params] = [Non-enmbedding ap]")
    print(f"Compressor AP [M]: {round(compressor_ap/10**6, 2)} - {round(compressor_ap_e/10**6, 2)} = {round(compressor_ap/10**6 - compressor_ap_e/10**6, 2)}")
    print(f"Default AP [M]: {round(default_ap/10**6, 2)} - {round(default_ap_e/10**6, 2)} = {round(default_ap/10**6 - default_ap_e/10**6, 2)}")
    print(f"Doner AP [M]: {round(doner_ap/10**6, 2)} - {round(doner_ap_e/10**6, 2)} = {round(doner_ap/10**6 - doner_ap_e/10**6, 2)}")
    print(f"Compression ratio: {round(default_ap/doner_ap, 3)}; {round(doner_ap/10**6, 2)} -> {round(default_ap/10**6, 2)}")

print(f"Compression (layres: {N_LAYERS}) {DM} -> {N_LAYERS*52}")
compress_info(DM, N_LAYERS*52, N_LAYERS)
print()

print(f"Compression (layres: {N_LAYERS}) {DM} -> {N_LAYERS*48}")
compress_info(DM, N_LAYERS*48, N_LAYERS)
print()

print(f"Compression (layres: {N_LAYERS}) {DM} -> {N_LAYERS*42}")
compress_info(DM, N_LAYERS*42, N_LAYERS)
print()

print(f"Compression (layres: {N_LAYERS}) {DM} -> {N_LAYERS*32}")
compress_info(DM, N_LAYERS*32, N_LAYERS)
print()



Compression (layres: 24) 1536 -> 1248
N layers: 24, doner dmodel: 1536, compressed dmodel: 1248, head/layers: 52.0
[Model]: [Act. params] - [Embedding act. params] = [Non-enmbedding ap]
Compressor AP [M]: 557.83 - 5.75 = 552.08
Default AP [M]: 349.08 - 124.8 = 224.28
Doner AP [M]: 493.34 - 153.6 = 339.74
Compression ratio: 0.708; 493.34 -> 349.08

Compression (layres: 24) 1536 -> 1152
N layers: 24, doner dmodel: 1536, compressed dmodel: 1152, head/layers: 48.0
[Model]: [Act. params] - [Embedding act. params] = [Non-enmbedding ap]
Compressor AP [M]: 514.92 - 5.31 = 509.61
Default AP [M]: 306.3 - 115.2 = 191.1
Doner AP [M]: 493.34 - 153.6 = 339.74
Compression ratio: 0.621; 493.34 -> 306.3

Compression (layres: 24) 1536 -> 1008
N layers: 24, doner dmodel: 1536, compressed dmodel: 1008, head/layers: 42.0
[Model]: [Act. params] - [Embedding act. params] = [Non-enmbedding ap]
Compressor AP [M]: 450.55 - 4.64 = 445.91
Default AP [M]: 247.11 - 100.8 = 146.31
Doner AP [M]: 493.34 - 153.6 = 339.